# Case 3: 3D Facies Modeling (LA3D)

This notebook demonstrates both unconditional and conditional diffusion for 3D facies modeling.

Volume size: [32, 48, 48] (D, H, W)

For visualization, we show 2D sections (slices) rather than 3D cubes.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from diffsim.models import Unet3D
from diffsim.core.diffusion import Diffusion
from diffsim.core.network import Network
from diffsim.data.dataset import NPYInpaintDataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def set_seed(seed=42):
    """Set random seed for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"Random seed set to {seed}")

def set_device(data):
    """Move tensor to device."""
    if isinstance(data, dict):
        return {k: set_device(v) for k, v in data.items()}
    elif isinstance(data, torch.Tensor):
        return data.to(device)
    return data

In [ ]:
# Load config
with open('../configs/case3_la3d.json') as f:
    config = json.load(f)
print(f"Loaded config for: {config['name']}")
print(f"Volume size: {config['image_size']}")

## Section 1: Unconditional Generation

In [ ]:
# Build unconditional 3D model
uncond = config['unconditional']
model = Unet3D(
    dim=uncond['dim'],
    channels=uncond['channels'],
    dim_mults=tuple(uncond['dim_mults'])
)
model.to(device)

# Load checkpoint
ckpt_path = Path('..') / config['checkpoints']['unconditional']
if ckpt_path.exists():
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"Loaded checkpoint from {ckpt_path}")
else:
    print(f"Checkpoint not found at {ckpt_path}. Using random weights.")

In [ ]:
# Initialize diffusion
diffusion = Diffusion(
    timesteps=uncond['timesteps'],
    beta_schedule=uncond['beta_schedule']
)

# Generate 3D samples using DDPM
set_seed(42)
image_size = tuple(config['image_size'])  # (D, H, W)
model.eval()
with torch.no_grad():
    samples_ddpm = diffusion.sample(
        model,
        image_size=image_size,
        batch_size=4,
        channels=uncond['channels']
    )

In [ ]:
# Visualize DDPM 3D samples - XY sections at different depths
n_samples = min(4, samples_ddpm.shape[0])
depth = samples_ddpm.shape[2]

fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4*n_samples))
slice_indices = [0, depth//3, 2*depth//3, depth-1]

for i in range(n_samples):
    for j, z in enumerate(slice_indices):
        ax = axes[i, j] if n_samples > 1 else axes[j]
        ax.imshow(samples_ddpm[i, 0, z].cpu().numpy(), cmap='viridis', vmin=0, vmax=1)
        ax.set_title(f'z={z}' if i == 0 else '')
        ax.axis('off')

plt.suptitle('DDPM 3D Samples (XY sections at different depths)', fontsize=16)
plt.tight_layout()
plt.show()

### DDIM Sampling (Faster)

In [ ]:
# Generate 3D samples using DDIM (faster, 50 steps instead of 1500)
set_seed(42)
model.eval()
with torch.no_grad():
    samples_ddim = diffusion.sample_ddim(
        model,
        image_size=image_size,
        batch_size=4,
        channels=uncond['channels'],
        ddim_steps=50,
        eta=0.0  # deterministic
    )

In [ ]:
# Visualize DDIM 3D samples - XY sections at different depths
n_samples = min(4, samples_ddim.shape[0])
depth = samples_ddim.shape[2]

fig, axes = plt.subplots(n_samples, 4, figsize=(16, 4*n_samples))
slice_indices = [0, depth//3, 2*depth//3, depth-1]

for i in range(n_samples):
    for j, z in enumerate(slice_indices):
        ax = axes[i, j] if n_samples > 1 else axes[j]
        ax.imshow(samples_ddim[i, 0, z].cpu().numpy(), cmap='viridis', vmin=0, vmax=1)
        ax.set_title(f'z={z}' if i == 0 else '')
        ax.axis('off')

plt.suptitle('DDIM 3D Samples (XY sections, 50 steps)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Show orthogonal sections for first DDPM sample
sample = samples_ddpm[0, 0].cpu().numpy()
d, h, w = sample.shape

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# XY section (top view)
axes[0].imshow(sample[d//2], cmap='viridis')
axes[0].set_title('XY section (z=mid)')
axes[0].set_xlabel('X')
axes[0].set_ylabel('Y')

# XZ section (front view)
axes[1].imshow(sample[:, h//2, :], cmap='viridis', aspect='auto')
axes[1].set_title('XZ section (y=mid)')
axes[1].set_xlabel('X')
axes[1].set_ylabel('Z')

# YZ section (side view)
axes[2].imshow(sample[:, :, w//2], cmap='viridis', aspect='auto')
axes[2].set_title('YZ section (x=mid)')
axes[2].set_xlabel('Y')
axes[2].set_ylabel('Z')

plt.suptitle('Orthogonal sections of DDPM 3D sample', fontsize=16)
plt.tight_layout()
plt.show()

## Section 2: Conditional Generation (3D Inpainting)

The conditional model takes sparse observations as conditioning input.

In [ ]:
# Build conditional 3D network
cond = config['conditional']
unet_config = {
    "image_size": config['image_size'][0],  # D dimension
    "in_channel": cond['in_channel'],
    "out_channel": cond['out_channel'],
    "inner_channel": cond['inner_channel'],
    "channel_mults": cond['channel_mults'],
    "attn_res": cond['attn_res'],
    "num_head_channels": cond['num_head_channels'],
    "res_blocks": cond['res_blocks'],
    "dropout": cond['dropout'],
}

# Use 3D module and get predict_type
module_name = cond.get('module_name', 'guided_diffusion_3d')
predict_type = cond.get('predict_type', 'epsilon')  # 'epsilon' or 'x_start'

network = Network(
    unet=unet_config,
    beta_schedule=cond['beta_schedule'],
    module_name=module_name,
    predict_type=predict_type
)
network.to(device)

print(f"Using module: {module_name}, predict_type: {predict_type}")

# Load checkpoint first with strict=False (ignore buffer mismatches)
ckpt_path = Path('..') / config['checkpoints']['conditional']
if ckpt_path.exists():
    network.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False), strict=False)
    print(f"Loaded checkpoint from {ckpt_path}")
else:
    print(f"Checkpoint not found at {ckpt_path}. Using random weights.")

# Setup noise schedule for test (use 200 steps for faster sampling)
# This will re-register the buffers with the correct size
test_schedule = {
    "schedule": cond['beta_schedule']['test']['schedule'],
    "n_timestep": 500,  # Reduced from 1500 for faster sampling
    "linear_start": cond['beta_schedule']['test']['linear_start'],
    "linear_end": cond['beta_schedule']['test']['linear_end']
}
network.beta_schedule['test'] = test_schedule
network.set_new_noise_schedule(device=torch.device(device), phase='test')

print(f"Using {network.num_timesteps} timesteps for sampling")

### Load 3D Dataset

In [ ]:
# Define data paths from config (support both old and new config structure)
cond_data = config.get('conditional', {}).get('data', config.get('data', {}))
images_path = cond_data.get('test_image', cond_data.get('test_image_path', ''))
masks_path = cond_data.get('test_mask', cond_data.get('test_mask_path', ''))
data_root = (images_path, masks_path)

# Mask configuration
mask_config = {
    'mask_mode': 'file'  # Use mask files from the masks directory
}

# Load dataset
dataset = NPYInpaintDataset(data_root, mask_config=mask_config, image_size=config['image_size'])
print(f"Dataset size: {len(dataset)}")

### Generate Multiple Realizations

In [ ]:
# Test indices for generation
numlist = [407, 489, 430, 61, 463]
num_realizations = 5

d, h, w = config['image_size']

print(f"Generating {num_realizations} realizations for {len(numlist)} test samples...")

In [ ]:
# Generate realizations for each test sample
set_seed(42)
network.eval()

# Store all outputs: [num_samples, num_realizations, D, H, W]
total_output = np.zeros((len(numlist), num_realizations, d, h, w))
gt_volumes = []
mask_volumes = []

for idx, inum in enumerate(numlist):
    print(f"Processing sample {idx+1}/{len(numlist)} (index {inum})...")
    
    # Get data from dataset
    data = dataset[inum]
    gt_image = data['gt_image']  # [1, D, H, W]
    mask = data['mask']  # [1, D, H, W]
    cond_image = data['cond_image']  # [5, D, H, W] - 5 conditioning channels
    
    gt_volumes.append(gt_image.cpu().numpy())
    mask_volumes.append(mask.cpu().numpy())
    
    # Prepare batched inputs
    cond_input = torch.from_numpy(cond_image).unsqueeze(0).repeat(num_realizations, 1, 1, 1, 1).to(device)
    gt_batch = gt_image.unsqueeze(0).repeat(num_realizations, 1, 1, 1, 1).to(device)
    mask_input = mask.unsqueeze(0).repeat(num_realizations, 1, 1, 1, 1).to(device)
    
    # Generate fresh noise for each realization
    yt_input = gt_image * (1. - mask) + mask * torch.randn(num_realizations, 1, d, h, w)
    yt_input = yt_input.to(device)
    
    with torch.no_grad():
        output, visuals = network.restoration(
            y_cond=cond_input, 
            y_t=yt_input,
            y_0=gt_batch, 
            mask=mask_input, 
            sample_num=8
        )
    
    total_output[idx, :, :, :, :] = output.cpu().numpy().reshape(num_realizations, d, h, w)

print("Generation complete!")
print(f"Output shape: {total_output.shape}")

### Plot Results: XY Section

In [ ]:
# Create output directory
dir_name = '../results/case3'
Path(dir_name).mkdir(parents=True, exist_ok=True)

# For each sample, show one XY section at mid-depth
# Columns: GT, Conditioning, Realizations...
n_samples = len(numlist)
mid_z = 10

for sample_idx in range(n_samples):
    fig, axes = plt.subplots(1, 2 + num_realizations, figsize=(3*(2+num_realizations), 3))
    
    gt = gt_volumes[sample_idx][0]  # [D, H, W]
    mask = mask_volumes[sample_idx][0]  # [D, H, W]
    outputs = total_output[sample_idx]  # [num_realizations, D, H, W]
    
    # GT
    axes[0].imshow(gt[mid_z], cmap='viridis')
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')
    
    # Conditioned (GT with mask applied)
    cond_view = gt[mid_z].copy()
    cond_view[mask[mid_z] > 0.5] = np.nan
    axes[1].imshow(cond_view, cmap='viridis')
    axes[1].set_title('Conditioning')
    axes[1].axis('off')
    
    # Realizations
    for r in range(num_realizations):
        axes[2+r].imshow(outputs[r, mid_z], cmap='viridis')
        axes[2+r].set_title(f'Real {r+1}')
        axes[2+r].axis('off')
    
    plt.suptitle(f'Sample {sample_idx}: XY section at z={mid_z}', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{dir_name}/sample_{sample_idx}_xy_section.png', dpi=200)
    plt.show()

print(f"Figures saved to {dir_name}/")

### Plot Results: XZ Section (Cross-section)

In [ ]:
# Show XZ section at y-position with well observations for each sample
for sample_idx in range(n_samples):
    fig, axes = plt.subplots(1, 2 + num_realizations, figsize=(3*(2+num_realizations), 2))
    
    gt = gt_volumes[sample_idx][0]  # [D, H, W]
    mask = mask_volumes[sample_idx][0]  # [D, H, W]
    outputs = total_output[sample_idx]  # [num_realizations, D, H, W]
    
    # Find y-slice with most well observations (where mask == 0)
    well_counts_per_y = np.sum(mask < 0.5, axis=(0, 2))  # sum over D and W
    best_y = np.argmax(well_counts_per_y)
    
    # GT (XZ section)
    axes[0].imshow(gt[:, best_y, :], cmap='viridis', aspect='auto')
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')
    
    # Conditioned
    cond_view = gt[:, best_y, :].copy()
    cond_view[mask[:, best_y, :] > 0.5] = np.nan
    axes[1].imshow(cond_view, cmap='viridis', aspect='auto')
    axes[1].set_title('Conditioning')
    axes[1].axis('off')
    
    # Realizations
    for r in range(num_realizations):
        axes[2+r].imshow(outputs[r, :, best_y, :], cmap='viridis', aspect='auto')
        axes[2+r].set_title(f'Real {r+1}')
        axes[2+r].axis('off')
    
    plt.suptitle(f'Sample {sample_idx}: XZ section at y={best_y}', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{dir_name}/sample_{sample_idx}_xz_section.png', dpi=200)
    plt.show()

print(f"Figures saved to {dir_name}/")

### DDIM Sampling for Conditional Generation (Faster)

In [ ]:
# Generate realizations using DDIM (faster, 50 steps instead of 200)
set_seed(42)
network.eval()

# Store all outputs: [num_samples, num_realizations, D, H, W]
total_output_ddim = np.zeros((len(numlist), num_realizations, d, h, w))

print(f"Generating {num_realizations} realizations for {len(numlist)} test samples using DDIM...")

for idx, inum in enumerate(numlist):
    print(f"Processing sample {idx+1}/{len(numlist)} (index {inum})...")
    
    # Get data from dataset
    data = dataset[inum]
    gt_image = data['gt_image']  # [1, D, H, W]
    mask = data['mask']  # [1, D, H, W]
    cond_image = data['cond_image']  # [5, D, H, W] - 5 conditioning channels
    
    # Prepare batched inputs
    cond_input = torch.from_numpy(cond_image).unsqueeze(0).repeat(num_realizations, 1, 1, 1, 1).to(device)
    gt_batch = gt_image.unsqueeze(0).repeat(num_realizations, 1, 1, 1, 1).to(device)
    mask_input = mask.unsqueeze(0).repeat(num_realizations, 1, 1, 1, 1).to(device)
    
    # Generate fresh noise for each realization
    yt_input = gt_image * (1. - mask) + mask * torch.randn(num_realizations, 1, d, h, w)
    yt_input = yt_input.to(device)
    
    with torch.no_grad():
        output, visuals = network.restoration_ddim(
            y_cond=cond_input, 
            y_t=yt_input,
            y_0=gt_batch, 
            mask=mask_input, 
            ddim_steps=50,  # Much faster than 200 DDPM steps
            eta=0.0,        # Deterministic (set eta>0 for stochasticity)
            sample_num=8
        )
    
    total_output_ddim[idx, :, :, :, :] = output.cpu().numpy().reshape(num_realizations, d, h, w)

print("DDIM Generation complete!")
print(f"Output shape: {total_output_ddim.shape}")

### Plot DDIM Results: XY Section

In [ ]:
# Visualize DDIM results: XY section at mid-depth
for sample_idx in range(n_samples):
    fig, axes = plt.subplots(1, 2 + num_realizations, figsize=(3*(2+num_realizations), 3))
    
    gt = gt_volumes[sample_idx][0]  # [D, H, W]
    mask = mask_volumes[sample_idx][0]  # [D, H, W]
    outputs = total_output_ddim[sample_idx]  # [num_realizations, D, H, W]
    
    # GT
    axes[0].imshow(gt[mid_z], cmap='viridis')
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')
    
    # Conditioned (GT with mask applied)
    cond_view = gt[mid_z].copy()
    cond_view[mask[mid_z] > 0.5] = np.nan
    axes[1].imshow(cond_view, cmap='viridis')
    axes[1].set_title('Conditioning')
    axes[1].axis('off')
    
    # Realizations
    for r in range(num_realizations):
        axes[2+r].imshow(outputs[r, mid_z], cmap='viridis')
        axes[2+r].set_title(f'Real {r+1}')
        axes[2+r].axis('off')
    
    plt.suptitle(f'DDIM Sample {sample_idx}: XY section at z={mid_z} (50 steps)', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{dir_name}/sample_{sample_idx}_xy_section_ddim.png', dpi=200)
    plt.show()

print(f"DDIM figures saved to {dir_name}/")

### Plot DDIM Results: XZ Section (Cross-section)

In [ ]:
# Visualize DDIM results: XZ section at y-position with well observations
for sample_idx in range(n_samples):
    fig, axes = plt.subplots(1, 2 + num_realizations, figsize=(3*(2+num_realizations), 2))
    
    gt = gt_volumes[sample_idx][0]  # [D, H, W]
    mask = mask_volumes[sample_idx][0]  # [D, H, W]
    outputs = total_output_ddim[sample_idx]  # [num_realizations, D, H, W]
    
    # Find y-slice with most well observations (where mask == 0)
    well_counts_per_y = np.sum(mask < 0.5, axis=(0, 2))  # sum over D and W
    best_y = np.argmax(well_counts_per_y)
    
    # GT (XZ section)
    axes[0].imshow(gt[:, best_y, :], cmap='viridis', aspect='auto')
    axes[0].set_title('Ground Truth')
    axes[0].axis('off')
    
    # Conditioned
    cond_view = gt[:, best_y, :].copy()
    cond_view[mask[:, best_y, :] > 0.5] = np.nan
    axes[1].imshow(cond_view, cmap='viridis', aspect='auto')
    axes[1].set_title('Conditioning')
    axes[1].axis('off')
    
    # Realizations
    for r in range(num_realizations):
        axes[2+r].imshow(outputs[r, :, best_y, :], cmap='viridis', aspect='auto')
        axes[2+r].set_title(f'Real {r+1}')
        axes[2+r].axis('off')
    
    plt.suptitle(f'DDIM Sample {sample_idx}: XZ section at y={best_y} (50 steps)', fontsize=14)
    plt.tight_layout()
    plt.savefig(f'{dir_name}/sample_{sample_idx}_xz_section_ddim.png', dpi=200)
    plt.show()

print(f"DDIM figures saved to {dir_name}/")